In [1]:
import os
import itertools
import multiprocessing as mp
import shutil

from concurrent.futures import ProcessPoolExecutor
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import supervision as sv

from trackers import ByteTrackTracker
from trackers.eval import evaluate_mot_sequences


# Detect the MOT17 root that contains `MOT17_yolox_dets`
BASE_DIR = Path.cwd()
while not (BASE_DIR / "MOT17_yolox_dets").exists() and BASE_DIR.parent != BASE_DIR:
    BASE_DIR = BASE_DIR.parent
if not (BASE_DIR / "MOT17_yolox_dets").exists():
    raise FileNotFoundError("Could not find 'MOT17_yolox_dets' in current or parent directories.")

MOT17_DET_ROOT = str((BASE_DIR / "MOT17_yolox_dets").resolve())

# GT aligned to YOLOX val frame ranges
MOT17_GT_YOLOX_VAL_ROOT = str((BASE_DIR / "TrackEval" / "data" / "gt" / "MOT17_yolox_val").resolve())
MOT17_GT_TRAINVAL = os.path.join(MOT17_GT_YOLOX_VAL_ROOT, "train_val")
MOT17_VAL_SEQMAP = str((BASE_DIR / "TrackEval" / "data" / "gt" / "MOT17" / "MOT17-val.txt").resolve())

MIN_BOX_AREA = 10
VERTICAL_RATIO_THRESH = 1.6


def build_dets_index(det_list):
    dets_by_frame = defaultdict(list)
    for line in det_list:
        frame_id = int(line.split(",")[0])
        dets_by_frame[frame_id].append(line)
    return dets_by_frame


def get_detections_from_dict(frame_id, dets_by_frame):
    dets = []
    for line in dets_by_frame.get(frame_id, []):
        det = line.split(",")
        x1 = float(det[1])
        y1 = float(det[2])
        x2 = float(det[3])
        y2 = float(det[4])
        conf = float(det[5])
        dets.append([x1, y1, x2, y2, conf])
    return dets


def write_mot_output(out_path: str):
    """Convert `MOT17-XX.txt` files into MOT17-server format (same as SORT notebook)."""
    out_dir = Path(out_path)

    existing = ["01", "03", "06", "07", "08", "12", "14"]
    missing = ["02", "04", "05", "09", "10", "11", "13"]
    suffixes = ["FRCNN", "SDP", "DPM"]

    for num in existing:
        src = out_dir / f"MOT17-{num}.txt"
        if not src.exists():
            print(f"Missing expected source file: {src}")
            continue
        content = src.read_bytes()
        for suf in suffixes:
            dst = out_dir / f"MOT17-{num}-{suf}.txt"
            dst.write_bytes(content)
        src.unlink()
        print(f"Replicated + removed: {src.name}")

    for num in missing:
        for suf in suffixes:
            dst = out_dir / f"MOT17-{num}-{suf}.txt"
            dst.touch(exist_ok=True)

    print("Empty MOT17 server files created for missing sequences.")


In [ ]:
def run_bytetrack_mot17_once(
    split: str,
    lost_track_buffer: int,
    track_activation_threshold: float,
    minimum_consecutive_frames: int,
    minimum_iou_threshold: float,
    high_conf_det_threshold: float,
    use_defaults: bool = False,
):
    """Run ByteTrack on one MOT17 split (train/val/test) for a given hyperparameter set."""

    assert split in {"train", "val", "test"}, f"Unsupported split: {split}"

    if use_defaults:
        tracker = ByteTrackTracker()
    else:
        tracker = ByteTrackTracker(
            lost_track_buffer=lost_track_buffer,
            track_activation_threshold=track_activation_threshold,
            minimum_consecutive_frames=minimum_consecutive_frames,
            minimum_iou_threshold=minimum_iou_threshold,
            high_conf_det_threshold=high_conf_det_threshold,
        )

    det_root = os.path.join(MOT17_DET_ROOT, split)

    outputs_root = "BYTETRACK_outputs_MOT17"
    os.makedirs(outputs_root, exist_ok=True)

    if use_defaults:
        save_dir = os.path.join(outputs_root, f"{split}_defaults")
    else:
        save_dir = os.path.join(
            outputs_root,
            f"{split}_L{lost_track_buffer}_A{track_activation_threshold}_"
            f"C{minimum_consecutive_frames}_I{minimum_iou_threshold}_H{high_conf_det_threshold}",
        )
    os.makedirs(save_dir, exist_ok=True)

    for seq in sorted(os.listdir(det_root)):
        if not seq.endswith(".txt"):
            continue

        tracker.reset()
        seq_name_raw = os.path.splitext(seq)[0]
        if split in {"train", "val"}:
            seq_name = seq_name_raw.split("_")[0] + "-FRCNN"
        else:
            seq_name = seq_name_raw

        with open(os.path.join(det_root, seq), "r") as f_det:
            det_list = f_det.readlines()
            dets_by_frame = build_dets_index(det_list)

        last_frame = int(det_list[-1].split(",")[0])
        output_lines = []

        for frame_id in range(1, last_frame + 1):
            raw_dets = get_detections_from_dict(frame_id, dets_by_frame)
            if raw_dets:
                raw_dets = np.array(raw_dets)
                dets = sv.Detections(xyxy=raw_dets[:, :4], confidence=raw_dets[:, 4])
            else:
                dets = sv.Detections.empty()

            dets = tracker.update(detections=dets)
            for tid, (left, top, right, bottom) in zip(dets.tracker_id, dets.xyxy):
                if tid == -1:
                    continue
                width = right - left
                height = bottom - top
                vertical = width / max(height, 1e-6) > VERTICAL_RATIO_THRESH
                if width * height > MIN_BOX_AREA and not vertical:
                    output_lines.append(
                        f"{frame_id},{int(tid)},{round(left,1)},{round(top,1)},{round(width,1)},{round(height,1)},-1,-1,-1,-1\n"
                    )

        save_path = os.path.join(save_dir, seq_name + ".txt")
        with open(save_path, "w") as f:
            f.writelines(output_lines)

        print(f"Finished {split} sequence {seq_name} -> {save_path}")

    HOTA = IDF1 = MOTA = None
    if split == "test":
        write_mot_output(save_dir)

    if split == "val":
        try:
            result = evaluate_mot_sequences(
                gt_dir=MOT17_GT_TRAINVAL,
                tracker_dir=save_dir,
                seqmap=MOT17_VAL_SEQMAP,
                metrics=["CLEAR", "HOTA", "Identity"],
            )
            agg = result.to_dict()["aggregate"]
            MOTA = agg["CLEAR"]["MOTA"]
            HOTA = agg["HOTA"]["HOTA"]
            IDF1 = agg["Identity"]["IDF1"]

            if use_defaults:
                print(f"split={split} | defaults -> HOTA: {HOTA:.3f}, IDF1: {IDF1:.3f}, MOTA: {MOTA:.3f} (results in {save_dir})")
            else:
                print(
                    f"split={split} | L:{lost_track_buffer}, "
                    f"A:{track_activation_threshold}, C:{minimum_consecutive_frames}, "
                    f"I:{minimum_iou_threshold}, H:{high_conf_det_threshold} -> "
                    f"HOTA: {HOTA:.3f}, IDF1: {IDF1:.3f}, MOTA: {MOTA:.3f} (results in {save_dir})"
                )
        except Exception as e:
            if use_defaults:
                print(f"split={split} | defaults -> evaluation FAILED ({repr(e)}). Results in {save_dir}")
            else:
                print(
                    f"split={split} | L:{lost_track_buffer}, "
                    f"A:{track_activation_threshold}, C:{minimum_consecutive_frames}, "
                    f"I:{minimum_iou_threshold}, H:{high_conf_det_threshold} -> "
                    f"evaluation FAILED ({repr(e)}). Results in {save_dir}"
                )

    return {
        "split": split,
        "lost_track_buffer": lost_track_buffer,
        "track_activation_threshold": track_activation_threshold,
        "minimum_consecutive_frames": minimum_consecutive_frames,
        "minimum_iou_threshold": minimum_iou_threshold,
        "high_conf_det_threshold": high_conf_det_threshold,
        "HOTA": HOTA,
        "IDF1": IDF1,
        "MOTA": MOTA,
        "output_dir": save_dir,
    }


In [ ]:
# Run ByteTrack with default parameters (ByteTrackTracker()) on the definitive eval set (val)
res = run_bytetrack_mot17_once("test", 0, 0.0, 0.0, 0.0, 0.0, use_defaults=True)
# Create a zip archive of the test outputs for submission
submission_dir = res["output_dir"]
submission_zip_base = os.path.basename(submission_dir)
submission_zip = "ByteTrack_" + submission_zip_base + ".zip"
shutil.make_archive("ByteTrack_" + submission_zip_base, "zip", submission_dir)

Finished test sequence MOT17-01 -> BYTETRACK_outputs_MOT17/test_defaults/MOT17-01.txt
Finished test sequence MOT17-03 -> BYTETRACK_outputs_MOT17/test_defaults/MOT17-03.txt
Finished test sequence MOT17-06 -> BYTETRACK_outputs_MOT17/test_defaults/MOT17-06.txt
Finished test sequence MOT17-07 -> BYTETRACK_outputs_MOT17/test_defaults/MOT17-07.txt
Finished test sequence MOT17-08 -> BYTETRACK_outputs_MOT17/test_defaults/MOT17-08.txt
Finished test sequence MOT17-12 -> BYTETRACK_outputs_MOT17/test_defaults/MOT17-12.txt
Finished test sequence MOT17-14 -> BYTETRACK_outputs_MOT17/test_defaults/MOT17-14.txt
Replicated + removed: MOT17-01.txt
Replicated + removed: MOT17-03.txt
Replicated + removed: MOT17-06.txt
Replicated + removed: MOT17-07.txt
Replicated + removed: MOT17-08.txt
Replicated + removed: MOT17-12.txt
Replicated + removed: MOT17-14.txt
Empty MOT17 server files created for missing sequences.


'/Users/alexanderbodner/Documents/roboflow/trackers metrics/mot17/ByteTrack_test_defaults.zip'

In [ ]:
lost_track_buffer_candidates = [10, 30, 60, 90]
track_activation_threshold_candidates = [0.2, 0.5, 0.7, 0.9]
minimum_consecutive_frames_candidates = [1, 3]
minimum_iou_threshold_candidates = [0.05, 0.1, 0.3]
high_conf_det_threshold_candidates = [0.5, 0.6, 0.7, 0.9]

all_candidate_lists = [
    lost_track_buffer_candidates,
    track_activation_threshold_candidates,
    minimum_consecutive_frames_candidates,
    minimum_iou_threshold_candidates,
    high_conf_det_threshold_candidates,
]

parameter_combinations = list(itertools.product(*all_candidate_lists))
print(f"Total ByteTrack parameter combinations for MOT17: {len(parameter_combinations)}")


Total ByteTrack parameter combinations for MOT17: 384


In [ ]:
def tune_bytetrack_on_mot17_val_parallel(parameter_combinations):
    """Run ByteTrack over MOT17 **val** for many hyperparameter sets in parallel."""

    num_workers = os.cpu_count()
    print(f"Running {len(parameter_combinations)} ByteTrack combinations on MOT17-val with {num_workers} workers")

    ctx = mp.get_context("fork")
    all_results = []

    with ProcessPoolExecutor(max_workers=num_workers, mp_context=ctx) as executor:
        futures = []
        for i, comb in enumerate(parameter_combinations):
            print(f"Submitting combination {i + 1}/{len(parameter_combinations)}")
            (
                lost_track_buffer,
                track_activation_threshold,
                minimum_consecutive_frames,
                minimum_iou_threshold,
                high_conf_det_threshold,
            ) = comb
            futures.append(
                executor.submit(
                    run_bytetrack_mot17_once,
                    "val",
                    lost_track_buffer,
                    track_activation_threshold,
                    minimum_consecutive_frames,
                    minimum_iou_threshold,
                    high_conf_det_threshold,
                )
            )

        for i, f in enumerate(futures):
            try:
                result = f.result()
                all_results.append(result)
                print(f"[{i + 1}/{len(parameter_combinations)}] val combination finished.")
            except Exception as e:
                print(f"[{i + 1}/{len(parameter_combinations)}] val combination FAILED: {repr(e)}")

    df = pd.DataFrame(all_results)
    print("Validation hyperparameter sweep complete. Results stored in 'bytetrack_MOT17_val_tuning_df'.")
    print(df.head())

    if not df.empty:
        df.to_csv("bytetrack_MOT17_val_tuning.csv", index=False)
    else:
        print("Warning: no successful validation runs. Check paths and errors printed above.")

    return df


In [ ]:
bytetrack_MOT17_val_tuning_df = tune_bytetrack_on_mot17_val_parallel(parameter_combinations)

if (
    not bytetrack_MOT17_val_tuning_df.empty
    and "HOTA" in bytetrack_MOT17_val_tuning_df.columns
    and not bytetrack_MOT17_val_tuning_df["HOTA"].isna().all()
):
    best_idx = bytetrack_MOT17_val_tuning_df["HOTA"].idxmax()
    best_row = bytetrack_MOT17_val_tuning_df.loc[best_idx]
    print("\nBest validation row by HOTA:\n", best_row)

    best_params = dict(
        lost_track_buffer=int(best_row["lost_track_buffer"]),
        track_activation_threshold=float(best_row["track_activation_threshold"]),
        minimum_consecutive_frames=int(best_row["minimum_consecutive_frames"]),
        minimum_iou_threshold=float(best_row["minimum_iou_threshold"]),
        high_conf_det_threshold=float(best_row["high_conf_det_threshold"]),
    )
    print("\nBest hyperparameters (val):", best_params)
else:
    print(
        "\nNo HOTA column found in 'bytetrack_MOT17_val_tuning_df' – you can still inspect 'output_dir' per row and evaluate externally."
    )


Running 384 ByteTrack combinations on MOT17-val with 10 workers
Submitting combination 1/384
Submitting combination 2/384
Submitting combination 3/384
Submitting combination 4/384
Submitting combination 5/384
Submitting combination 6/384
Submitting combination 7/384
Submitting combination 8/384
Submitting combination 9/384
Submitting combination 10/384
Submitting combination 11/384
Submitting combination 12/384
Submitting combination 13/384
Submitting combination 14/384
Submitting combination 15/384
Submitting combination 16/384
Submitting combination 17/384
Submitting combination 18/384
Submitting combination 19/384
Submitting combination 20/384
Submitting combination 21/384
Submitting combination 22/384
Submitting combination 23/384
Submitting combination 24/384
Submitting combination 25/384
Submitting combination 26/384
Submitting combination 27/384
Submitting combination 28/384
Submitting combination 29/384
Submitting combination 30/384
Submitting combination 31/384
Submitting comb

In [6]:
if "best_params" in globals():
    print("Using MOT17-test ByteTrack hyperparameters:", best_params)

    test_result = run_bytetrack_mot17_once(
        split="test",
        **best_params,
    )

    print("\nTest split metrics (if GT available):")
    print({k: test_result.get(k) for k in ["HOTA", "IDF1", "MOTA"]})

    # Create a zip archive of the test outputs for submission
    submission_dir = test_result["output_dir"]
    submission_zip_base = os.path.basename(submission_dir)
    submission_zip = "ByteTrack_" + submission_zip_base + ".zip"
    shutil.make_archive("ByteTrack_" + submission_zip_base, "zip", submission_dir)
    print("\nCreated MOT17 submission archive:", submission_zip)
    print("Archive contains MOT17 server-format txt files ready for upload.")
else:
    print("No best_params found – run the validation sweep cell first.")


Using MOT17-test ByteTrack hyperparameters: {'lost_track_buffer': 10, 'frame_rate': 30.0, 'track_activation_threshold': 0.7, 'minimum_consecutive_frames': 1, 'minimum_iou_threshold': 0.3, 'high_conf_det_threshold': 0.5}
Finished test sequence MOT17-01 -> BYTETRACK_outputs_MOT17/test_L10_F30.0_A0.7_C1_I0.3_H0.5/MOT17-01.txt
Finished test sequence MOT17-03 -> BYTETRACK_outputs_MOT17/test_L10_F30.0_A0.7_C1_I0.3_H0.5/MOT17-03.txt
Finished test sequence MOT17-06 -> BYTETRACK_outputs_MOT17/test_L10_F30.0_A0.7_C1_I0.3_H0.5/MOT17-06.txt
Finished test sequence MOT17-07 -> BYTETRACK_outputs_MOT17/test_L10_F30.0_A0.7_C1_I0.3_H0.5/MOT17-07.txt
Finished test sequence MOT17-08 -> BYTETRACK_outputs_MOT17/test_L10_F30.0_A0.7_C1_I0.3_H0.5/MOT17-08.txt
Finished test sequence MOT17-12 -> BYTETRACK_outputs_MOT17/test_L10_F30.0_A0.7_C1_I0.3_H0.5/MOT17-12.txt
Finished test sequence MOT17-14 -> BYTETRACK_outputs_MOT17/test_L10_F30.0_A0.7_C1_I0.3_H0.5/MOT17-14.txt
Replicated + removed: MOT17-01.txt
Replicated